# Train thêm âm thanh cho Urban Sound trên Google Colab

Chạy lần lượt từng ô (Shift+Enter). Không cần cài gì trên máy. Nên bật **Runtime → Change runtime type → T4 GPU** cho nhanh (CPU vẫn chạy được).

Kết quả cuối cùng: file `models/custom/head.json` – commit lên GitHub là web tự dùng.

## 1. Lấy mã nguồn
Thay link bằng repo của nhóm. Repo riêng tư thì dùng link có token: `https://<TOKEN>@github.com/<nhom>/urban-sound.git`.

In [ ]:
REPO = 'https://github.com/<ten-nhom>/urban-sound.git'  # <- sửa
!git clone -q $REPO urban-sound
%cd urban-sound/training
!pip -q install -r requirements.txt yt-dlp
!apt-get -qq install -y ffmpeg > /dev/null

## 2. (Tùy chọn) Đưa mẫu tự thu lên
Tải gói .zip từ trang **Đóng góp dữ liệu** rồi upload ở ô dưới.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
os.makedirs('data/my_sounds', exist_ok=True)
for name in up:
    zipfile.ZipFile(name).extractall('data/my_sounds')
!ls data/my_sounds

## 3. Tải dữ liệu còn thiếu
Sửa danh sách trong `sounds_to_collect.json` nếu muốn thêm âm thanh. Freesound cần API key miễn phí: https://freesound.org/apiv2/apply

In [ ]:
import os
os.environ['FREESOUND_API_KEY'] = 'DAN_KEY_VAO_DAY'  # <- sửa
!python 00_download_freesound.py --n 150

In [ ]:
# AudioSet (YouTube). Thêm --unbalanced để có nhiều mẫu hơn (lâu hơn)
!python 00_download_audioset.py --n 150 --workers 8

In [ ]:
# Tra tên lớp AudioSet khi muốn thêm nhãn mới, ví dụ:
!python 00_download_audioset.py --list bell

## 4. (Tùy chọn) UrbanSound8K và ESC-50
ESC-50 nhỏ (600 MB) tải được trực tiếp. UrbanSound8K cần tải từ trang chính thức rồi upload/mount Google Drive.

In [ ]:
!wget -q https://github.com/karolpiczak/ESC-50/archive/master.zip -O esc50.zip && unzip -q esc50.zip -d /content/
ESC = '/content/ESC-50-master'

## 5. Nghe thử và xóa mẫu sai
Dữ liệu tải tự động luôn có mẫu sai nhãn. Nghe ngẫu nhiên vài file mỗi nhãn, xóa file sai.

In [ ]:
import random, glob, IPython.display as ipd
for lab in sorted(os.listdir('data/extra')):
    fs = glob.glob(f'data/extra/{lab}/*')
    if not fs: continue
    f = random.choice(fs); print(lab, len(fs), f)
    display(ipd.Audio(f))

## 6. Gom dữ liệu → trích embedding → train

In [ ]:
!python 01_prepare_data.py --esc50 $ESC --max-per-label 600
# có UrbanSound8K thì thêm: --us8k /content/drive/MyDrive/UrbanSound8K

In [ ]:
!python 02_extract_embeddings.py --augment 3

In [ ]:
!python 03_train_head.py

## 7. Lấy kết quả
Tải `head.json` về rồi đặt vào `models/custom/` trong repo và commit. Hoặc push thẳng từ Colab (cần token).

In [ ]:
files.download('../models/custom/head.json')
files.download('data/report.json')